# Extracción del precio de Activos Crypto en tiempo real con Binance

In [1]:
## uv add WebSocket-client
import websocket, json

In [11]:
# formato para Raw-Streams: wss://stream.binance.com:9443/ws/<symbol>t@kline_<interval>

crypto_symbol = 'btcusdt'
interval = '1m'

socket = f'wss://stream.binance.com:9443/ws/{crypto_symbol}@kline_{interval}'

In [7]:
socket

'wss://stream.binance.com:9443/ws/btcusdt@kline_1m'

In [19]:
# logs
def on_message(wsapp, message):
    print(message)

def on_open(ws):
    print(f"{'#' * 30} Opened connection {'#' * 30}\n")

def on_close(ws, close_status_code, close_msg):
    print(f"\n{'#' * 30} Closed connection {'#' * 30}")

In [20]:
ws = websocket.WebSocketApp(socket, on_open=on_open, on_message=on_message, on_close=on_close)

# 📊 Datos del Payload

## Evento principal

| Clave | Valor          | Descripción       |
|-------|----------------|-------------------|
| e     | kline          | Tipo de evento    |
| E     | 1672515782136  | Hora del evento   |
| s     | BNBBTC         | Símbolo           |

---

## Objeto `k` (Kline)

| Clave | Valor          | Descripción                          |
|-------|----------------|--------------------------------------|
| t     | 1672515780000  | Hora de inicio del kline             |
| T     | 1672515839999  | Hora de cierre del kline             |
| s     | BNBBTC         | Símbolo                              |
| i     | 1m             | Intervalo                            |
| f     | 100            | ID de la primera operación           |
| L     | 200            | ID de la última operación            |
| o     | 0.0010         | Precio de apertura                   |
| c     | 0.0020         | Precio de cierre                     |
| h     | 0.0025         | Precio más alto                      |
| l     | 0.0015         | Precio más bajo                      |
| v     | 1000           | Volumen del activo base              |
| n     | 100            | Número de operaciones                |
| x     | false          | ¿Está cerrado este kline?            |
| q     | 1.0000         | Volumen del activo cotizado          |
| V     | 500            | Volumen de compra de taker (base)    |
| Q     | 0.500          | Volumen de compra de taker (quote)   |
| B     | 123456         | Ignorar                              |

In [21]:
ws.run_forever()

############################## Opened connection ##############################

{"e":"kline","E":1771804684030,"s":"BTCUSDT","k":{"t":1771804680000,"T":1771804739999,"s":"BTCUSDT","i":"1m","f":5995994937,"L":5995994943,"o":"67714.31000000","c":"67714.31000000","h":"67714.31000000","l":"67714.30000000","v":"0.15130000","n":7,"x":false,"q":"10245.17507720","V":"0.14872000","Q":"10070.47218320","B":"0"}}
{"e":"kline","E":1771804686018,"s":"BTCUSDT","k":{"t":1771804680000,"T":1771804739999,"s":"BTCUSDT","i":"1m","f":5995994937,"L":5995995056,"o":"67714.31000000","c":"67715.08000000","h":"67715.09000000","l":"67714.30000000","v":"0.45135000","n":120,"x":false,"q":"30562.89461600","V":"0.44846000","Q":"30367.20004720","B":"0"}}
{"e":"kline","E":1771804688017,"s":"BTCUSDT","k":{"t":1771804680000,"T":1771804739999,"s":"BTCUSDT","i":"1m","f":5995994937,"L":5995995084,"o":"67714.31000000","c":"67715.09000000","h":"67715.09000000","l":"67714.30000000","v":"0.45662000","n":148,"x":false,"q":"3091

True

# Crear Velas

In [25]:
closes, highs, lows = [], [], []

In [30]:
# logs
def on_message(ws, message):
    json_message = json.loads(message)
    candle = json_message['k']
    is_candle_closed = candle['x']
    close = candle['c']
    high = candle['h']
    low = candle['l']
    vol = candle['v']

    if is_candle_closed:
        closes.append(float(close))
        highs.append(float(high))
        lows.append(float(low))

        print(closes)
        print(highs)
        print(lows)

def on_open(ws):
    print(f"{'#' * 30} Opened connection {'#' * 30}\n")

def on_close(ws, close_status_code, close_msg):
    print(f"\n{'#' * 30} Closed connection {'#' * 30}")

In [31]:
ws = websocket.WebSocketApp(socket, on_open=on_open, on_message=on_message, on_close=on_close)

In [32]:
ws.run_forever()

############################## Opened connection ##############################

[67244.3]
[67265.87]
[67205.78]
[67244.3, 67250.64]
[67265.87, 67260.58]
[67205.78, 67244.3]
[67244.3, 67250.64, 67199.01]
[67265.87, 67260.58, 67250.65]
[67205.78, 67244.3, 67190.18]

############################## Closed connection ##############################


True

# Gráficos Dinámicos

In [12]:
import plotly.graph_objects as go
from IPython.display import clear_output

In [13]:
opens, closes, highs, lows, volumes = [], [], [], [], []

In [14]:
def update_plot():
    clear_output(wait=True)

    fig = go.Figure(data=[go.Candlestick(
        x=list(range(len(closes))),  # índice de cada vela
        open=opens,
        high=highs,
        low=lows,
        close=closes,
        name="Velas japonesas"
    )])

    fig.update_layout(
        title="Gráfica dinámica de velas japonesas",
        xaxis_title="Candlestick #",
        yaxis_title="Precio",
        template="plotly_dark"
    )

    fig.show()

def on_message(ws, message):
    json_message = json.loads(message)
    candle = json_message['k']
    is_candle_closed = candle['x']
    open_ = candle['o']
    close = candle['c']
    high = candle['h']
    low = candle['l']
    vol = candle['v']

    if is_candle_closed:
        opens.append(float(open_))
        closes.append(float(close))
        highs.append(float(high))
        lows.append(float(low))
        volumes.append(float(vol))

        print("Cierres:", closes)
        print("Altos:", highs)
        print("Bajos:", lows)

        # Actualizamos la gráfica cada vez que se cierra una vela
        update_plot()

def on_open(ws):
    print(f"{'#' * 30} Opened connection {'#' * 30}\n")

def on_close(ws, close_status_code, close_msg):
    print(f"\n{'#' * 30} Closed connection {'#' * 30}")

In [15]:
ws = websocket.WebSocketApp(socket, on_message=on_message, on_open=on_open , on_close=on_close)

In [16]:
ws.run_forever()


############################## Closed connection ##############################


True